# 11장 실습 — 시뮬레이션 루프 직접 짜기 (채점)

직접 짜는 셋 중 마지막입니다. 채울 파일은 **`labs/ch11_simloop.py`** 입니다.

지금까지 만든 것을 전부 이어 붙입니다.
3장의 최단경로가 소요시간을 주고, 8장의 수요가 호출을 만들고, 10장의 배차가 차를 고릅니다.
이 장에서 만드는 것은 그것들을 1분마다 부르는 바깥 루프입니다.

채점 기준은 실제 DTUMOS 엔진과의 거리입니다. 평균 대기시간이 1.5분 이내로 붙어야 합니다.
완전히 같을 수는 없습니다. 어디서 왜 갈라지는지를 설명할 수 있으면 됩니다.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

import ch11_simloop as sol      # 여러분이 채우는 파일

## 1. 한 스텝에 무엇을 하는가 (교재 11.1)

1분마다 이 다섯 가지를 순서대로 합니다.

1. 이 분에 들어온 호출을 대기열에 넣습니다
2. 너무 오래 기다린 승객을 실패로 처리합니다
3. 대기 승객과 빈 차로 비용행렬을 만들어 배차합니다
4. 배차된 차의 도착 시각을 정합니다
5. 이 분의 상태를 한 줄로 기록합니다

순서가 중요합니다. 접수보다 배차를 먼저 하면 방금 들어온 호출이 1분 늦게 처리됩니다.

In [ ]:
from smartmob.data import load_demand, load_vehicles

demand = load_demand("hanam")
vehicles = load_vehicles("hanam")

print(f"호출 {len(demand):,}건, 차량 {len(vehicles)}대")
print(f"차량 컬럼: {list(vehicles.columns)}")
vehicles.head(3)

차량마다 `work_start` 와 `work_end` 가 있습니다.
"80대"가 항상 80대가 아니라는 뜻입니다. 0장에서 본 그 현상의 원인입니다.

## 2. 작게 먼저 돌려 봅니다

6시간을 한 번에 돌리지 말고 1시간만 돌려 봅니다.
틀렸을 때 어디가 틀렸는지 보이는 크기에서 시작합니다.

In [ ]:
banner("18:00 ~ 19:00 (60분)")
try:
    small = sol.simulate(demand, vehicles, 1080, 1140)
    print(small.record.head())
    print()
    print(small.summary())
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

## 3. 전체 구간

In [ ]:
import time

banner("18:00 ~ 24:00 (360분)")
try:
    t0 = time.perf_counter()
    run = sol.simulate(demand, vehicles, 1080, 1440)
    print(f"{time.perf_counter() - t0:.1f}초")

    expect("record 행 수", len(run.record), 360)
    expect("record 컬럼", list(run.record.columns),
           ["time", "waiting_passenger_cnt", "fail_passenger_cnt",
            "empty_vehicle_cnt", "driving_vehicle_cnt"])

    s = run.summary()
    print(f"    서비스율 {s['service_rate']:.1%}, 평균대기 {s['avg_waiting_time_min']:.2f}분")
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

## 4. 엔진과 맞춰 보기 (교재 11.5)

같은 수요와 같은 차량으로 DTUMOS 를 돌린 결과가 저장소에 녹화되어 있습니다.
두 결과를 나란히 놓습니다.

In [ ]:
from smartmob import Dtumos

engine = Dtumos().run_simulation(
    city="hanam", mode="taxi", fleet_size=80, num_passengers=1000,
    time_start=1080, time_end=1440, dispatch_mode="optimization",
    matrix_mode="street_distance", vehicle_capacity=1, random_seed=42,
)
engine.summary()

In [ ]:
import pandas as pd

from smartmob.teaching.metrics import kpi_table

try:
    run = sol.simulate(demand, vehicles, 1080, 1440)
    table = pd.DataFrame({
        "내 루프": kpi_table(run),
        "DTUMOS": kpi_table(engine),
    })
    display(table.round(3))
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

값이 완전히 같지는 않습니다. 다른 것이 정상입니다. 이유는 크게 셋입니다.

1. 소요시간 모형이 다릅니다. 내 루프는 직선거리 기준이고 엔진은 도로망 위를 달립니다
2. 승하차 시간의 처리가 다릅니다
3. 엔진은 차량이 목적지에 도착한 뒤의 위치를 정확히 추적합니다

**숫자가 다른 것 자체는 문제가 아닙니다. 왜 다른지 모르는 것이 문제입니다.**

## 5. 채점

In [ ]:
from check import check

try:
    report = check("ch11")
except NotImplementedError as exc:
    print("아직 구현하지 않았습니다 —", exc)

## 6. 조건을 바꿔 보기 (교재 11.7)

루프가 있으면 "만약에"를 물을 수 있습니다. 그것이 시뮬레이터를 만드는 이유입니다.
아래는 교재의 정돈본으로 돌립니다. 여러분 것이 완성되면 `sol.simulate` 로 바꿔 보세요.

In [ ]:
from smartmob.teaching.simloop import simulate as reference

rows = []
for n_vehicles in [20, 40, 60, 80, 120, 160]:
    r = reference(demand, vehicles.head(n_vehicles), 1080, 1440)
    s = r.summary()
    rows.append({
        "차량": n_vehicles,
        "서비스율": round(s["service_rate"], 3),
        "평균대기_분": round(s["avg_waiting_time_min"], 2),
        "최대대기_분": round(s["max_waiting_time_min"], 1),
        "가동률": round(s["utilization"], 3),
    })

sweep = pd.DataFrame(rows)
sweep

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(sweep["차량"], sweep["평균대기_분"], marker="o", color="#4C6EF5", label="평균 대기 (분)")
ax1.set_xlabel("차량 대수")
ax1.set_ylabel("평균 대기 (분)", color="#4C6EF5")

ax2 = ax1.twinx()
ax2.plot(sweep["차량"], sweep["가동률"], marker="s", color="crimson", label="가동률")
ax2.set_ylabel("가동률", color="crimson")

ax1.set_title("차량을 늘리면 승객은 좋아지고 차량은 논다")
plt.tight_layout();

승객과 사업자의 이해가 정반대로 움직입니다.
어느 지점을 고를지는 계산이 아니라 판단입니다. 12장에서 그 판단을 표로 만듭니다.

## 7. 배차 방법을 바꿔 보기 (교재 11.8)

In [ ]:
rows = []
for n_vehicles in [20, 40, 80]:
    for match in ["greedy", "optimal"]:
        s = reference(demand, vehicles.head(n_vehicles), 1080, 1440, match=match).summary()
        rows.append({
            "차량": n_vehicles,
            "배차": match,
            "평균대기_분": round(s["avg_waiting_time_min"], 2),
            "서비스율": round(s["service_rate"], 3),
        })

pd.DataFrame(rows).pivot(index="차량", columns="배차", values="평균대기_분")

차량이 넉넉하면 두 방법의 차이가 거의 없습니다.
**배차 알고리즘은 수요가 공급을 압박할 때만 의미가 있습니다.**
이것이 실무 프로젝트에서 다시 나옵니다.

## 제출할 것

1. 채운 `labs/ch11_simloop.py`
2. 채점 셀의 출력 (전부 PASS)
3. 막혔던 지점과 어떻게 풀었는지 3~5줄
4. **내 루프와 엔진이 왜 완전히 같지 않은지** 한 문단

4번이 이 과제의 핵심입니다.

## 정리

- 루프는 1분마다 접수·포기·배차·이동·기록을 순서대로 합니다. 순서가 결과를 바꿉니다
- 차량 대수는 상수가 아닙니다. 근무 시간이 있습니다
- 엔진과 값이 다른 것은 정상입니다. 이유를 아는 것이 목표입니다
- 12장 실습에서는 이 결과를 보고서에 쓸 표와 그림으로 만듭니다